The goal is to:
- Learn patterns between resumes and job descriptions
- Predict candidate suitability
- Compare multiple models and select the best one


In [25]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import os

In [11]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

In [27]:
import os

# Check what's saved in data folder
print("Files in data folder:")
for f in os.listdir("../data"):
    print(f)

# Check files in models folder    
print("\nFiles in models folder:")
for f in os.listdir("../models"):
    print(f)

Files in data folder:
label_distribution.png
resume_jd_test.csv
resume_jd_test_cleaned.csv
resume_jd_train.csv
resume_jd_train_cleaned.csv
similarity_by_label.png
test_jd_embeddings.npy
test_resume_embeddings.npy
tfidf_vs_bert.png
train_jd_embeddings.npy
train_resume_embeddings.npy
X_test.npz
X_train.npz
y_test.npy
y_train.npy

Files in models folder:
bert_model.pkl
best_model.pkl
label_encoder.pkl
tfidf_vectorizer.pkl


In [28]:
import numpy as np
import scipy.sparse as sp
import joblib

# Load TF-IDF features
X_train_tfidf = sp.load_npz("../data/X_train.npz")
X_test_tfidf  = sp.load_npz("../data/X_test.npz")

# Load BERT embeddings
train_resume_bert = np.load("../data/train_resume_embeddings.npy")
train_jd_bert     = np.load("../data/train_jd_embeddings.npy")
test_resume_bert  = np.load("../data/test_resume_embeddings.npy")
test_jd_bert      = np.load("../data/test_jd_embeddings.npy")

# Compute BERT cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

train_bert_sim = np.array([
    cosine_similarity(
        train_resume_bert[i].reshape(1,-1),
        train_jd_bert[i].reshape(1,-1)
    )[0][0]
    for i in range(len(train_resume_bert))
])

test_bert_sim = np.array([
    cosine_similarity(
        test_resume_bert[i].reshape(1,-1),
        test_jd_bert[i].reshape(1,-1)
    )[0][0]
    for i in range(len(test_resume_bert))
])

# Convert BERT features to sparse
train_bert_sparse = sp.csr_matrix(
    np.hstack([
        train_resume_bert,
        train_jd_bert,
        train_bert_sim.reshape(-1,1)
    ])
)

test_bert_sparse = sp.csr_matrix(
    np.hstack([
        test_resume_bert,
        test_jd_bert,
        test_bert_sim.reshape(-1,1)
    ])
)

# Combine TF-IDF + BERT
X_train = sp.hstack([X_train_tfidf, train_bert_sparse])
X_test  = sp.hstack([X_test_tfidf,  test_bert_sparse])

# Load labels
y_train = np.load("../data/y_train.npy")
y_test  = np.load("../data/y_test.npy")
le      = joblib.load("../models/label_encoder.pkl")

print("✅ All features combined!")
print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

✅ All features combined!
X_train shape: (6240, 10770)
X_test shape:  (1759, 10770)


c:\Users\reach\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [15]:
## Training Models

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        solver='lbfgs'   # ✅ supports multiclass automatically
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
}

The models are trained on:
- TF-IDF vectors of combined resume and job description text
- These vectors represent important keywords in numerical form

In [30]:
results = {}

for name, model in models.items():
    print(f"\n🔹 Training {name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')

    results[name] = {
        'model': model,
        'accuracy': acc,
        'f1': f1,
        'y_pred': y_pred
    }

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")


🔹 Training Logistic Regression...
Accuracy: 0.5412
F1 Score: 0.5150

🔹 Training Random Forest...
Accuracy: 0.5259
F1 Score: 0.4176

🔹 Training XGBoost...


c:\Users\reach\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [08:03:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.5560
F1 Score: 0.5336


In [31]:
print(results.keys()) 

dict_keys(['Logistic Regression', 'Random Forest', 'XGBoost'])


We trained three different machine learning models on the TF-IDF features extracted from the cleaned dataset:

- Logistic Regression  
- Random Forest  
- XGBoost  

### Training Results

| Model                 | Accuracy | F1 Score |
|----------------------|---------:|---------:|
| Logistic Regression  | 0.5196   | 0.4837   |
| Random Forest        | 0.5355   | 0.4414   |
| XGBoost              | 0.5185   | 0.4899   |

### Model Selection

Although Random Forest achieved the highest accuracy, we selected **XGBoost** as the best model because it achieved the highest **F1 Score (0.4899)**.

F1 Score is preferred over accuracy because:
- It balances precision and recall
- It is more reliable for imbalanced datasets
- It ensures better performance across all classes

In [32]:
import joblib
import os

# create folder
os.makedirs("../models", exist_ok=True)

# select best model (you already know XGBoost is best)
best_model = results["XGBoost"]['model']

# save it
joblib.dump(best_model, "../models/best_model.pkl")

print("✅ Best model saved!")

✅ Best model saved!


In [33]:
X_train.shape

(6240, 10770)